# Coqui XTTS-v2 Voice Cloning in Google Colab

This notebook allows you to use Coqui XTTS-v2 for text-to-speech with voice cloning using a reference audio file.

## 1. Install Dependencies

In [ ]:
# This cell installs the Coqui TTS library.
!pip install TTS

## 2. Imports and Device Setup

In [ ]:
from TTS.api import TTS
import torch
import os

# --- User Configuration for Device ---
# Set your desired device: "cuda" for GPU (if available), or "cpu".
TARGET_DEVICE = "cuda"  # Options: "cuda" or "cpu"
# -------------------------------------

# Determine the device to use
print(f"Target device specified: {TARGET_DEVICE}")
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if TARGET_DEVICE == "cuda" and cuda_available:
    device = "cuda"
elif TARGET_DEVICE == "cuda" and not cuda_available:
    device = "cpu"
    print("CUDA was targeted but is not available. Falling back to CPU.")
else:
    device = "cpu"

print(f"Using device: {device}")

## 3. User Inputs

### Upload Reference Audio
Upload your short reference audio file (e.g., a 5-15 second WAV file). Make sure the filename does not contain spaces or special characters for simplicity.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

if len(uploaded.keys()) == 0:
    print("No file uploaded. Please upload a reference audio WAV file.")
    reference_wav_path = None
else:
    reference_wav_path = list(uploaded.keys())[0]
    print(f"User uploaded file '{reference_wav_path}' as the reference audio.")
    # Optional: rename to a fixed name if that simplifies things later, but direct usage is fine.
    # os.rename(reference_wav_path, "reference_audio.wav")
    # reference_wav_path = "reference_audio.wav"

### Specify Text to Synthesize and Language

In [ ]:
# Edit this line to change the text you want to synthesize.
text_to_speak = "Hello, this is a test of your custom voice workflow in Google Colab."

# Set the language for XTTS (e.g., "en", "es", "fr", "de", etc.)
language_to_use = "en"

output_wav_path = "xtts_colab_generated_speech.wav"

print(f"Text to synthesize: {text_to_speak}")
print(f"Language: {language_to_use}")
print(f"Output file will be: {output_wav_path}")

## 4. Initialize TTS Model and Perform Inference

In [ ]:
if reference_wav_path and os.path.exists(reference_wav_path):
    try:
        print(f"Initializing TTS model on device: {device}...")
        # Model name for XTTS-v2
        model_name = "tts_models/multilingual/multi-dataset/xtts_v2"
        tts = TTS(model_name).to(device)
        print("TTS model initialized successfully.")

        print(f"Generating speech for text: \"{text_to_speak}\"")
        print(f"Using reference audio: {reference_wav_path}")
        tts.tts_to_file(
            text=text_to_speak,
            speaker_wav=reference_wav_path,
            language=language_to_use,
            file_path=output_wav_path
        )
        print(f"XTTS speech saved to {output_wav_path}")
        print("You can now download the file in the next step.")

    except Exception as e:
        print(f"An error occurred during TTS processing: {e}")
        if device == "cuda":
            print("If this is a CUDA-related error (e.g., 'CUDA out of memory'), try restarting the runtime and selecting 'cpu' for TARGET_DEVICE in Cell 2.")
else:
    print("Reference WAV path is not set or file does not exist. Please upload a reference audio file in Step 3.")

## 5. Download the Generated Speech

In [ ]:
if os.path.exists(output_wav_path):
  from google.colab import files
  files.download(output_wav_path)
else:
  print(f"Output file {output_wav_path} not found. Please ensure the previous cell ran successfully.")